In [2]:
import pandas as pd
from pathlib import Path

In [8]:
DATA_PATH = Path("../data/processed/02_eda_data.parquet")

SPLIT_DATA = Path("../data/metadata/split_ledger.parquet")

In [9]:
df = pd.read_parquet(DATA_PATH)

In [10]:
def apply_split_reference(main_df: pd.DataFrame, ref_path: str, id_col: str):
    """
    Loads the split ledger and returns separated train and test DataFrames.
    """
    # Load the 2-column reference file
    split_ledger = pd.read_parquet(ref_path)
    
    # Merge the split assignment onto your main dataset
    # Using an inner merge ensures we only keep IDs that were officially split
    merged_df = main_df.merge(split_ledger, on=id_col, how='inner')
    
    # Separate the data
    train_df = merged_df[merged_df['split'] == 'train'].drop(columns=['split'])
    test_df = merged_df[merged_df['split'] == 'test'].drop(columns=['split'])
    
    return train_df, test_df

# Example Usage:


In [11]:
train_data, test_data = apply_split_reference(
    main_df=df,
    ref_path=SPLIT_DATA,
    id_col="user_id",
)

In [13]:
train_data.shape

(8276, 17)

In [14]:
SAFE_FEATURES = [
    "age",
    "employment_type",
    "monthly_income",
    "credit_score",
    "purchase_amount",
    "product_category",
    "bnpl_installments",
    "app_usage_frequency",
    "location",
    "transaction_date",
    "debt_to_income_ratio",
]

TARGET = "default_flag"

EXCLUDED_FEATURES = [
    "user_id",
    "repayment_delay_days",
    "risk_score",
    "customer_segment",
]


In [15]:
train_data["transaction_date"] = pd.to_datetime(
    train_data["transaction_date"],
    errors="raise"
)


In [16]:
train_data["transaction_year"] = train_data["transaction_date"].dt.year
train_data["transaction_month"] = train_data["transaction_date"].dt.month
train_data["transaction_dayofweek"] = train_data["transaction_date"].dt.dayofweek

train_data["is_weekend"] = (
    train_data["transaction_dayofweek"] >= 5
).astype(int)


In [17]:
train_data.drop(columns=["transaction_date"])

,user_id,age,employment_type,monthly_income,credit_score,purchase_amount,product_category,bnpl_installments,repayment_delay_days,missed_payments,default_flag,app_usage_frequency,location,debt_to_income_ratio,risk_score,customer_segment,transaction_year,transaction_month,transaction_dayofweek,is_weekend
1,2,19,Student,7247.85,300,1073.23,Fashion,12,13,1,0,3.09,USA,0.148076,266.0,High Risk,2024,10,0,0
2,3,20,Self-Employed,41582.26,471,5000.00,Electronics,3,19,2,0,3.33,Australia,0.120244,229.6,High Risk,2023,4,2,0
4,5,43,Salaried,42845.50,512,5000.00,Electronics,9,0,0,0,7.36,India,0.116698,135.2,High Risk,2024,10,5,1
5,6,52,Self-Employed,79950.32,729,5000.00,Fashion,9,1,0,0,6.32,Canada,0.062539,50.4,Low Risk,2024,8,3,0
6,7,44,Salaried,43192.27,511,5000.00,Sports,3,2,0,0,5.58,USA,0.115761,139.6,High Risk,2024,8,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10337,10338,39,Unemployed,13867.00,317,5000.00,Fashion,12,5,1,1,2.27,India,0.360568,243.2,High Risk,2024,8,3,0
10340,10341,52,Student,17344.34,393,5000.00,Fashion,6,0,0,1,4.33,USA,0.288278,182.8,High Risk,2024,10,4,0
10341,10342,53,Student,17788.20,300,100.00,Sports,12,12,1,0,1.41,Australia,0.005622,264.0,High Risk,2023,6,3,0
10342,10343,42,Self-Employed,59350.25,535,5000.00,Home,12,4,2,0,5.07,Australia,0.084246,174.0,High Risk,2024,9,1,0


In [18]:
import pandas as pd


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create deterministic features available at BNPL application time.
    """

    result = df.copy()

    # ------------------------------------------
    # Date features
    # ------------------------------------------

    result["transaction_date"] = pd.to_datetime(
        result["transaction_date"],
        errors="raise"
    )

    result["transaction_year"] = (
        result["transaction_date"].dt.year
    )

    result["transaction_month"] = (
        result["transaction_date"].dt.month
    )

    result["transaction_dayofweek"] = (
        result["transaction_date"].dt.dayofweek
    )

    result["is_weekend"] = (
        result["transaction_dayofweek"] >= 5
    ).astype(int)

    # ------------------------------------------
    # Affordability features
    # ------------------------------------------

    result["purchase_to_income_ratio"] = (
        result["purchase_amount"]
        / result["monthly_income"]
    )

    result["installment_amount"] = (
        result["purchase_amount"]
        / result["bnpl_installments"]
    )

    result["installment_to_income_ratio"] = (
        result["installment_amount"]
        / result["monthly_income"]
    )

    result["income_after_installment"] = (
        result["monthly_income"]
        - result["installment_amount"]
    )

    # Original timestamp has now served its purpose
    result = result.drop(columns=["transaction_date"])

    return result


In [20]:
engineered_data = engineer_features(train_data)

In [21]:
engineered_data.shape

(8276, 24)

In [22]:
engineered_data.to_parquet("../data/processed/03_engineered_data.parquet")